# RAG Pipeline — Gemini Q&A over Vertex AI Documentation

Builds a full Retrieval-Augmented Generation pipeline that answers questions about
Google Cloud / Vertex AI using the official documentation as its knowledge base.

**Backend toggle** — set once here, used throughout the notebook:
- `"gemini_api"` — free Gemini API (Days 2–3 development)
- `"vertex_ai"` — enterprise Vertex AI SDK (Day 4 verification)

**Sections:**
1. Setup & Auth
2. Corpus Ingestion
3. Chunking
4. Embedding
5. Retrieval
6. Generation
7. Interactive Q&A ← demo centerpiece
8. RAGAS Evaluation _(Day 3)_
9. Analysis _(Day 3)_

## Section 1 — Setup & Auth

In [4]:
import os
import sys
import time
import json
import subprocess
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

# ── Backend toggle ─────────────────────────────────────────────────────────────
BACKEND = "gemini_api"   # or "vertex_ai"
# ───────────────────────────────────────────────────────────────────────────────

load_dotenv(dotenv_path="../.env")
sys.path.insert(0, str(Path("../src").resolve()))

GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL  = "gemini-embedding-001"

if BACKEND == "gemini_api":
    from google import genai
    from google.genai import errors as genai_errors

    api_key = os.environ.get("GEMINI_API_KEY")
    assert api_key, "GEMINI_API_KEY not set — check .env"
    client = genai.Client(api_key=api_key)
    print(f"Backend  : Gemini API (free tier)")
    print(f"API key  : {api_key[:8]}...")

elif BACKEND == "vertex_ai":
    import vertexai
    from vertexai.generative_models import GenerativeModel
    from google.api_core import exceptions as gcp_errors

    project = os.environ.get("GCP_PROJECT_ID")
    location = os.environ.get("GCP_LOCATION", "us-central1")
    assert project, "GCP_PROJECT_ID not set — check .env"
    vertexai.init(project=project, location=location)
    client = None  # Vertex AI uses module-level calls
    print(f"Backend  : Vertex AI")
    print(f"Project  : {project}  Location: {location}")

else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}. Use 'gemini_api' or 'vertex_ai'.")

print(f"Generation model : {GENERATION_MODEL}")
print(f"Embedding model  : {EMBEDDING_MODEL}")

Backend  : Gemini API (free tier)
API key  : AIzaSyDJ...
Generation model : gemini-2.5-flash
Embedding model  : gemini-embedding-001


In [5]:
def with_retry(fn, retries=4, base_delay=5):
    """Retry fn on 503 UNAVAILABLE with exponential backoff."""
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if "503" not in str(e) and "UNAVAILABLE" not in str(e):
                raise
            if attempt == retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"503 UNAVAILABLE — retrying in {delay}s (attempt {attempt + 1}/{retries})...")
            time.sleep(delay)

print("Retry helper ready.")

Retry helper ready.


## Section 2 — Corpus Ingestion

Fetches and cleans GCP/Vertex AI documentation pages into `corpus/`.
The script (`src/build_corpus.py`) is idempotent — safe to re-run, skips nothing
already saved. Expect 15–30 seconds to fetch all pages with polite crawl delays.

Skip this cell if `corpus/manifest.json` already exists from a previous run.

In [7]:
REPO_ROOT = Path.cwd().parent  # Jupyter sets cwd to notebooks/
CORPUS_DIR = REPO_ROOT / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.json"

if MANIFEST_PATH.exists():
    print("Corpus already built — loading manifest.")
else:
    print("Building corpus (this might take a minute) ...")
    result = subprocess.run(
        [sys.executable, str(REPO_ROOT / "src" / "build_corpus.py")],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("build_corpus.py failed")

manifest = json.loads(MANIFEST_PATH.read_text())
pages = manifest["pages"]
skipped = manifest.get("skipped", [])

print(f"\nCorpus: {len(pages)} pages loaded, {len(skipped)} skipped")
for p in pages:
    print(f"  {p['slug']:<40} {p['chars']:>8,} chars")

Corpus already built — loading manifest.

Corpus: 19 pages loaded, 0 skipped
  vertex_ai_overview                          2,989 chars
  gemini_enterprise_platform                 15,949 chars
  gemini_models                              13,097 chars
  gemini_api_overview                        31,898 chars
  gemini_multimodal                           9,721 chars
  embeddings_overview                        16,866 chars
  embeddings_api_reference                   22,671 chars
  rag_overview                                6,215 chars
  rag_quickstart                              7,749 chars
  grounding_overview                          2,573 chars
  context_cache                               6,246 chars
  agent_builder_overview                      2,825 chars
  vertex_ai_search                           11,872 chars
  agent_builder_intro                         5,021 chars
  vector_search_overview                     17,156 chars
  gcp_auth_adc                                5,768 c

## Section 3 — Chunking

Split each corpus document into overlapping word-based chunks.
Each chunk carries: `text`, `source_url`, `chunk_index`, `doc_title`.

- **Chunk size:** 500 words  
- **Overlap:** 50 words (so adjacent chunks share context at boundaries)  
- **Min size:** 30 words (tail fragments below this are discarded)

In [8]:
from chunker import chunk_corpus
from collections import Counter

chunks = chunk_corpus(CORPUS_DIR, manifest)

print(f"Total chunks: {len(chunks)}")
print()

counts = Counter(c["doc_title"] for c in chunks)
for title, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {title:<45} {n:>4} chunks")

print()
print("--- Sample chunk ---")
sample = chunks[10]
print(f"doc_title  : {sample['doc_title']}")
print(f"source_url : {sample['source_url']}")
print(f"chunk_index: {sample['chunk_index']}")
print(f"word count : {len(sample['text'].split())}")
print(f"text       : {sample['text'][:300]}...")

Total chunks: 98

  Gemini Pricing                                  19 chunks
  Gemini API Overview                             13 chunks
  Embeddings API Reference                         8 chunks
  Vertex AI Access Control                         8 chunks
  Gemini Enterprise Platform                       6 chunks
  Embeddings Overview                              6 chunks
  Vector Search Overview                           6 chunks
  Gemini Models                                    5 chunks
  Vertex AI Search                                 5 chunks
  Gemini Multimodal                                4 chunks
  RAG Overview                                     3 chunks
  RAG Quickstart                                   3 chunks
  Context Cache                                    3 chunks
  Agent Builder Intro                              2 chunks
  GCP Auth ADC                                     2 chunks
  Responsible AI                                   2 chunks
  Vertex AI Overview  

## Section 4 — Embedding

Embed all corpus chunks with `gemini-embedding-001` (3072 dimensions) and cache to disk.
Uses `task_type=RETRIEVAL_DOCUMENT` for corpus chunks — the API optimises the vector
for asymmetric retrieval (document vs. query), which improves recall.

Re-runs are instant: if `corpus/embeddings.npy` already exists the cached file is loaded.

In [ ]:
from embedder import Embedder, load_embeddings

EMBEDDINGS_PATH = CORPUS_DIR / "embeddings.npy"
CHUNKS_PATH     = CORPUS_DIR / "chunks.json"

if EMBEDDINGS_PATH.exists() and CHUNKS_PATH.exists():
    print("Cached embeddings found — loading from disk.")
    embeddings, chunks = load_embeddings(CORPUS_DIR)
else:
    embedder = Embedder(client)
    embeddings = embedder.embed_and_save(chunks, CORPUS_DIR)
    _, chunks = load_embeddings(CORPUS_DIR)  # reload to confirm round-trip

print(f"\nembeddings shape : {embeddings.shape}")
print(f"chunks count     : {len(chunks)}")
print(f"vector norm[0]   : {float((embeddings[0]**2).sum()**0.5):.4f}")
print()
print("--- Sample chunk + vector head ---")
s = chunks[10]
print(f"doc_title  : {s['doc_title']}")
print(f"chunk_index: {s['chunk_index']}")
print(f"vector[:6] : {embeddings[10][:6].tolist()}")

## Section 5 — Retrieval

_Issue #9 — coming next._

In [ ]:
# TODO (#9): cosine similarity retrieval
print("Section 5 not yet implemented — see issue #9")

## Section 6 — Generation

_Issue #10 — coming next._

In [ ]:
# TODO (#10): RAG-augmented Gemini generation
print("Section 6 not yet implemented — see issue #10")

## Section 7 — Interactive Q&A

_Issue #11 — coming next. This is the demo centerpiece._

In [ ]:
# TODO (#11): ask(question) → retrieved chunks + generated answer
print("Section 7 not yet implemented — see issue #11")

## Section 8 — RAGAS Evaluation

_Issues #12–13 — Day 3._

In [ ]:
# TODO (#12, #13): RAGAS evaluation over test set
print("Section 8 not yet implemented — see issues #12–13")

## Section 9 — Analysis

_Issue #14 — Day 3 notebook polish._